# Writing Your First Agent Skill: From SKILL.md to AWS Agent Registry

This notebook walks through the full lifecycle of agent skills:

1. **Create** a skill from scratch (SKILL.md format)
2. **Load** skills using every method the Strands SDK provides
3. **Wire** skills into a live Strands Agent and watch it work
4. **Manage** skills at runtime (add, list, inspect activations)
5. **Register** skills in AWS Agent Registry and consume them three ways

**Prerequisites:**
- Python >= 3.10
- AWS credentials configured with Amazon Bedrock access (Claude 4.5 Haiku) and AgentCore access

In [ ]:
!pip install -q strands-agents strands-agents-tools mcp-proxy-for-aws -qU

---
## Part 1: Creating a Skill

A skill is a directory containing a `SKILL.md` file. The file has two parts:
- **YAML frontmatter** — metadata (name, description, allowed tools)
- **Markdown body** — instructions the agent follows when the skill is activated

The directory name **must match** the `name` field in the frontmatter.

Let's create an IaC security review skill that knows how to audit Terraform and CloudFormation files.

In [ ]:
import os
import shutil
from pathlib import Path

SKILL_DIR = Path("./demo-skills/iac-security-review")

SKILL_CONTENT = """\
---
name: iac-security-review
description: >
  Review infrastructure-as-code files (Terraform, CloudFormation) for security
  misconfigurations. Use when the user asks to review IaC, check for security
  issues in infrastructure code, or audit cloud resource definitions.
license: Apache-2.0
allowed-tools: file_read shell
metadata:
  author: platform-team
  version: \"1.0\"
---

You are an infrastructure security reviewer. When asked to review IaC files:

## Step 1: Identify IaC files

Use `file_read` or directory listing to find all `.tf`, `.yaml`, `.yml`,
and `.json` files that define cloud resources. Focus on the paths the user
specifies, or scan the current directory if none are given.

## Step 2: Check for common misconfigurations

For each file, check against these categories:

### Public access
- S3 buckets with public ACLs or missing `BlockPublicAccess`
- Security groups with `0.0.0.0/0` on sensitive ports (22, 3389, 3306, 5432)
- RDS instances with `publicly_accessible = true`
- API Gateway endpoints without authentication

### Encryption
- S3 buckets without server-side encryption
- EBS volumes without encryption
- RDS instances without `storage_encrypted`
- SNS/SQS without KMS encryption

### IAM
- Policies with `\\\"Action\\\": \\\"*\\\"` or `\\\"Resource\\\": \\\"*\\\"`
- Roles without condition keys
- Missing `MFA` conditions on sensitive operations

### Logging
- CloudTrail not enabled
- S3 access logging disabled
- VPC flow logs missing

## Step 3: Report findings

For each finding, report:
1. **File and line** — exact location
2. **Severity** — Critical, High, Medium, Low
3. **Issue** — what's wrong
4. **Fix** — specific code change to remediate

Sort findings by severity (Critical first). If no issues are found, say so
explicitly — don't invent problems.
"""

# Clean up previous runs
if Path("./demo-skills").exists():
    shutil.rmtree("./demo-skills")

# Create directory structure
SKILL_DIR.mkdir(parents=True)
(SKILL_DIR / "references").mkdir()
(SKILL_DIR / "scripts").mkdir()

# Write SKILL.md
(SKILL_DIR / "SKILL.md").write_text(SKILL_CONTENT)

# Show what we created
print("Skill directory structure:")
for root, dirs, files in os.walk(SKILL_DIR):
    level = root.replace(str(SKILL_DIR), "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

### Validate the skill with the Strands SDK

The `Skill.from_file()` method parses the SKILL.md, extracts the frontmatter, and validates the structure. If the name doesn't match the directory name or the frontmatter is malformed, it will raise an error (with `strict=True`) or a warning.

In [ ]:
from strands import Skill

skill = Skill.from_file("./demo-skills/iac-security-review")

print(f"Name:          {skill.name}")
print(f"Description:   {skill.description[:80]}...")
print(f"License:       {skill.license}")
print(f"Allowed tools: {skill.allowed_tools}")
print(f"Metadata:      {skill.metadata}")
print(f"Instructions:  {len(skill.instructions)} chars")
print(f"Path:          {skill.path}")

# Verify name matches directory
assert skill.name == SKILL_DIR.name, f"Name mismatch: {skill.name} != {SKILL_DIR.name}"
print("\n--- Skill parsed and validated successfully ---")

---
## Part 2: All the Ways to Load Skills

The Strands SDK gives you multiple ways to create `Skill` objects. Let's try each one.

### 2a. `Skill.from_content()` — parse from a string

Useful when the SKILL.md content comes from an API, database, or is generated dynamically.

In [ ]:
raw_md = """\
---
name: code-review
description: Review Python code for best practices, bugs, and security issues.
metadata:
  author: dev-team
  version: "1.0"
---

You are a senior Python code reviewer. When asked to review code:

1. Read all Python files in the specified path using `file_read`
2. Check for:
   - Security issues (SQL injection, command injection, hardcoded secrets)
   - Performance problems (N+1 queries, unnecessary loops)
   - Style violations (PEP 8, naming conventions)
   - Missing error handling
3. Report findings with file, line number, severity, and suggested fix
"""

skill_from_content = Skill.from_content(raw_md)
print(f"Name:         {skill_from_content.name}")
print(f"Description:  {skill_from_content.description}")
print(f"Path:         {skill_from_content.path}  (None — not loaded from disk)")
print(f"Instructions: {len(skill_from_content.instructions)} chars")

### 2b. `Skill()` constructor — create programmatically

For skills that are defined entirely in code, without a SKILL.md file.

In [ ]:
skill_programmatic = Skill(
    name="incident-response",
    description=(
        "Guide incident response for production outages. "
        "Use when the user reports a service disruption or needs help debugging a live issue."
    ),
    instructions=(
        "You are an incident commander. When activated:\n\n"
        "1. Assess severity (P1-P4) based on user impact\n"
        "2. Identify affected services from error logs or metrics\n"
        "3. Suggest immediate mitigation steps\n"
        "4. Draft a status page update\n"
        "5. Create a follow-up action items list"
    ),
    allowed_tools=["shell", "http_request"],
    metadata={"author": "sre-team", "version": "1.0"},
)

print(f"Name:          {skill_programmatic.name}")
print(f"Description:   {skill_programmatic.description[:70]}...")
print(f"Allowed tools: {skill_programmatic.allowed_tools}")
print(f"Instructions:  {len(skill_programmatic.instructions)} chars")

### 2c. `Skill.from_directory()` — bulk load from a parent directory

Scans a parent directory for all subdirectories containing a valid `SKILL.md` and returns a list of `Skill` objects. Let's create a second skill first so there's more to find.

In [ ]:
# Create a second skill so from_directory finds multiple
cost_dir = Path("./demo-skills/cost-optimization")
cost_dir.mkdir(parents=True, exist_ok=True)
(cost_dir / "SKILL.md").write_text("""\
---
name: cost-optimization
description: Analyze AWS costs and suggest optimizations. Use when asked about cloud spending, cost reduction, or resource right-sizing.
metadata:
  author: finops-team
  version: "1.0"
---

You are a FinOps advisor. When asked about AWS costs:

1. Review the resources described by the user
2. Identify over-provisioned or idle resources
3. Suggest Reserved Instances, Savings Plans, or Spot where applicable
4. Estimate potential savings as a percentage
""")

# Now load all skills from the parent directory
all_skills = Skill.from_directory("./demo-skills/")
print(f"Found {len(all_skills)} skills:")
for s in all_skills:
    print(f"  - {s.name}: {s.description[:60]}...")

### 2d. `AgentSkills` plugin — mix and match sources

The `AgentSkills` plugin accepts paths (strings), `Skill` objects, or a mix of both. It's the bridge between skills and the agent.

In [ ]:
from strands import AgentSkills

summarize_skill = Skill(
    name="summarize",
    description="Summarize long documents into key points",
    instructions="Read the document. Extract the 5 most important points.",
)

plugin = AgentSkills(
    skills=[
        "./demo-skills/iac-security-review",  # path to a single skill
        "./demo-skills/cost-optimization",     # another path
        summarize_skill,                        # programmatic skill
    ]
)

print(f"Plugin loaded {len(plugin.get_available_skills())} skills:")
for s in plugin.get_available_skills():
    print(f"  - {s.name}")

---
## Part 3: Wiring Skills Into a Live Agent

Now the fun part. Let's create a Strands Agent with the IaC security review skill and point it at some intentionally insecure Terraform.

Here's what happens at runtime:
1. The `AgentSkills` plugin injects skill names + descriptions (~100 tokens each) into the system prompt
2. The user's message arrives
3. The model recognizes the task matches a skill and calls the internal `skills` tool
4. Full SKILL.md instructions are loaded into context
5. The model follows the instructions, using the available tools

**Important:** Skills provide instructions, not tools. You must still give the agent the tools that the skill references (e.g., `file_read`, `shell`).

In [ ]:
# Create a sample Terraform file with intentional security issues
TERRAFORM_DIR = Path("./demo-terraform")
TERRAFORM_DIR.mkdir(parents=True, exist_ok=True)

(TERRAFORM_DIR / "main.tf").write_text("""\
# main.tf — intentionally insecure for demo purposes

provider "aws" {
  region = "us-east-1"
}

resource "aws_s3_bucket" "data_lake" {
  bucket = "company-data-lake-prod"
}

# Missing: no BlockPublicAccess, no encryption, no access logging

resource "aws_security_group" "web" {
  name        = "web-sg"
  description = "Web server security group"

  ingress {
    from_port   = 22
    to_port     = 22
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]  # SSH open to the world
  }

  ingress {
    from_port   = 443
    to_port     = 443
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  egress {
    from_port   = 0
    to_port     = 0
    protocol    = "-1"
    cidr_blocks = ["0.0.0.0/0"]
  }
}

resource "aws_db_instance" "main" {
  identifier     = "prod-database"
  engine         = "postgres"
  engine_version = "15.4"
  instance_class = "db.r6g.large"
  allocated_storage = 100

  publicly_accessible = true    # Database exposed to internet
  # Missing: storage_encrypted, backup_retention_period
}

resource "aws_iam_policy" "admin_like" {
  name = "too-permissive-policy"
  policy = jsonencode({
    Version = "2012-10-17"
    Statement = [
      {
        Effect   = "Allow"
        Action   = "*"          # Wildcard action
        Resource = "*"          # Wildcard resource
      }
    ]
  })
}
""")

print(f"Created insecure Terraform at: {TERRAFORM_DIR / 'main.tf'}")
print(f"\nContents:")
print((TERRAFORM_DIR / "main.tf").read_text())

In [ ]:
from strands import Agent, AgentSkills
from strands.models import BedrockModel
from strands_tools import file_read

# Set up the model (uses AWS credentials from environment)
model = BedrockModel(
    model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    region_name="us-west-2",
    max_tokens=4096,
)

# Load the security review skill
plugin = AgentSkills(skills="./demo-skills/iac-security-review")
print(f"Skills available: {[s.name for s in plugin.get_available_skills()]}")

# Create the agent
agent = Agent(
    model=model,
    plugins=[plugin],
    tools=[file_read],
    system_prompt="You are a helpful assistant with access to specialized skills.",
)

# Let the agent review the Terraform
terraform_path = TERRAFORM_DIR.resolve()
result = agent(f"Review the Terraform files in {terraform_path} for security issues.")
print(f"\nStop reason: {result.stop_reason}")

The agent:
1. Saw the `iac-security-review` skill description in its system prompt
2. Recognized the task was a match and called the `skills` tool to activate it
3. Loaded the full SKILL.md instructions
4. Followed the 3-step process (identify files, check misconfigurations, report findings)
5. Used `file_read` to examine the Terraform
6. Reported findings sorted by severity with specific remediation code

All of that behavior came from the skill's instructions — not from the system prompt or any hardcoded logic.

---
## Part 4: Runtime Skill Management

Skills aren't static. You can add, list, and inspect skills while the agent is running. This is useful for building agents that adapt their capabilities based on context — loading different skill sets based on the user's role, the project, or the task at hand.

In [ ]:
# Start fresh with the security skill
plugin = AgentSkills(skills="./demo-skills/iac-security-review")

print("Available skills (before):")
for s in plugin.get_available_skills():
    print(f"  - {s.name}: {s.description[:60]}...")

In [ ]:
# Add a cost estimation skill at runtime
cost_skill = Skill(
    name="cost-estimation",
    description=(
        "Estimate monthly AWS costs for a given architecture. "
        "Use when the user asks about pricing or cost estimates."
    ),
    instructions=(
        "You are an AWS cost estimation expert.\n\n"
        "When asked to estimate costs:\n"
        "1. Identify all AWS resources in the described architecture\n"
        "2. Look up approximate on-demand pricing for each resource\n"
        "3. Calculate monthly costs assuming 730 hours/month\n"
        "4. Present a table with: Resource, Type/Size, Unit Price, Monthly Cost\n"
        "5. Show the total and suggest cost optimization opportunities"
    ),
    metadata={"author": "finops-team", "version": "1.0"},
)

current = plugin.get_available_skills()
plugin.set_available_skills(current + [cost_skill])

print(f"Available skills (after adding):")
for s in plugin.get_available_skills():
    print(f"  - {s.name}")

In [ ]:
# Create agent with both skills and ask a cost question
# The agent should pick cost-estimation, NOT iac-security-review
agent = Agent(
    model=model,
    plugins=[plugin],
    tools=[file_read],
    system_prompt="You are a helpful cloud architecture assistant.",
)

result = agent(
    "Give me a quick cost estimate for running 3 t3.medium EC2 instances "
    "and 1 db.r6g.large RDS PostgreSQL instance in us-east-1. Keep it brief."
)
print(f"\nStop reason: {result.stop_reason}")

In [ ]:
# Inspect which skills were activated
activated = plugin.get_activated_skills(agent)
available = plugin.get_available_skills()

print(f"Total available:  {len(available)}")
print(f"Total activated:  {len(activated)}")
print(f"Activated skills: {activated}")
print(f"Still dormant:    {len(available) - len(activated)}")
print()
print("This demonstrates progressive disclosure — only the activated")
print("skill's full instructions were loaded into context. The other")
print("skill contributed only ~100 tokens (its name and description).")

---
## Part 5: Registering and Consuming Skills via AWS Agent Registry

So far everything has been local — skills on disk, loaded at startup. In a real organization, the platform team writes a security review skill, the FinOps team writes a cost optimization skill, and nobody knows what already exists.

[AWS Agent Registry](https://aws.amazon.com/bedrock/agentcore/) solves this. It's a governed catalog inside Amazon Bedrock AgentCore where teams register skills (and agents, MCP servers, etc.) and others discover them via semantic search.

The registry has two API planes:
- **Control plane** (`bedrock-agentcore-control`) — CRUD for registries and records
- **Data plane** (`bedrock-agentcore`) — `SearchRegistryRecords` with semantic search

And it exposes itself as an **MCP server**, so any MCP-compatible client (IDEs, Strands agents) can query it directly.

We'll demonstrate three consumption paths:
- **Path A:** `GetRegistryRecord` → `Skill.from_content()` (direct lookup)
- **Path B:** `SearchRegistryRecords` → `Skill.from_content()` (semantic discovery)
- **Path C:** MCP endpoint → lazy loading at runtime

### 5a. Create a registry and register our skill

In [ ]:
import json
import time
import boto3

REGION = "us-west-2"
ctrl = boto3.client("bedrock-agentcore-control", region_name=REGION)
dp = boto3.client("bedrock-agentcore", region_name=REGION)
account_id = boto3.client("sts").get_caller_identity()["Account"]

# Create a registry
print("Creating registry...")
resp = ctrl.create_registry(
    name="blog-demo-registry",
    description="Demo registry for blog post tutorial",
)
REGISTRY_ARN = resp["registryArn"]
REGISTRY_ID = REGISTRY_ARN.split("/")[-1]
print(f"  Registry ID: {REGISTRY_ID}")

# Wait for READY status
print("Waiting for registry to become READY...")
for _ in range(40):
    r = ctrl.get_registry(registryId=REGISTRY_ID)
    if r["status"] == "READY":
        print(f"  Status: {r['status']}")
        break
    time.sleep(3)

# Read the SKILL.md we created in Part 1
skill_md_content = (SKILL_DIR / "SKILL.md").read_text()

# Register the skill
print("\nRegistering skill...")
skill_definition = json.dumps({
    "repository": {
        "url": "https://github.com/my-org/agent-skills/tree/main/skills/iac-security-review",
        "source": "github",
    },
    "packages": [],
})

resp = ctrl.create_registry_record(
    registryId=REGISTRY_ID,
    name="iac-security-review",
    description="Review infrastructure-as-code files for security misconfigurations",
    descriptorType="AGENT_SKILLS",
    descriptors={
        "agentSkills": {
            "skillMd": {"inlineContent": skill_md_content},
            "skillDefinition": {"inlineContent": skill_definition},
        }
    },
    recordVersion="1.0",
)
RECORD_ID = resp["recordArn"].split("/")[-1]
print(f"  Record ID: {RECORD_ID}")
print(f"  Status: {resp['status']}")

# Wait for record to leave CREATING, then approve
for _ in range(20):
    rec = ctrl.get_registry_record(registryId=REGISTRY_ID, recordId=RECORD_ID)
    if rec["status"] != "CREATING":
        break
    time.sleep(2)

print("\nSubmitting for approval...")
ctrl.submit_registry_record_for_approval(registryId=REGISTRY_ID, recordId=RECORD_ID)
time.sleep(2)

print("Approving...")
ctrl.update_registry_record_status(
    registryId=REGISTRY_ID,
    recordId=RECORD_ID,
    status="APPROVED",
    statusReason="Approved for blog demo",
)

rec = ctrl.get_registry_record(registryId=REGISTRY_ID, recordId=RECORD_ID)
print(f"  Final status: {rec['status']}")

### 5b. Path A — `GetRegistryRecord` → `Skill.from_content()`

The simplest path. You know the record ID — fetch it, extract the SKILL.md, feed it to Strands. The full SKILL.md content comes back **inline** in the response — no second fetch needed.

In [ ]:
from strands import Skill

# Fetch the record
record = ctrl.get_registry_record(registryId=REGISTRY_ID, recordId=RECORD_ID)

# Extract the SKILL.md — it's inline in the response
skill_md = record["descriptors"]["agentSkills"]["skillMd"]["inlineContent"]
print(f"Record name:    {record['name']}")
print(f"Descriptor type: {record['descriptorType']}")
print(f"SKILL.md:       {len(skill_md)} chars")

# Load directly into Strands
skill_from_registry = Skill.from_content(skill_md)
print(f"\nLoaded into Strands:")
print(f"  Name:          {skill_from_registry.name}")
print(f"  Description:   {skill_from_registry.description[:60]}...")
print(f"  Instructions:  {len(skill_from_registry.instructions)} chars")
print(f"  Allowed tools: {skill_from_registry.allowed_tools}")

### 5c. Path B — `SearchRegistryRecords` → `Skill.from_content()`

When you don't know what skills exist, use the data plane's semantic search. Natural language queries return matching records with full descriptors — including the complete SKILL.md content.

In [ ]:
# Search index needs time after approval — retry until results appear
print("Waiting for search index...")
records = []
for attempt in range(6):
    time.sleep(10)
    results = dp.search_registry_records(
        registryIds=[REGISTRY_ARN],
        searchQuery="security review for Terraform",
        maxResults=5,
        filters={"descriptorType": {"$eq": "AGENT_SKILLS"}},
    )
    records = results.get("registryRecords", [])
    if records:
        break
    print(f"  Not indexed yet (attempt {attempt + 1}), retrying...")

print(f"Found {len(records)} skill(s):\n")

# Load each result into Strands
skills_from_search = []
for r in records:
    md = r["descriptors"]["agentSkills"]["skillMd"]["inlineContent"]
    s = Skill.from_content(md)
    skills_from_search.append(s)
    print(f"  - {r['name']} [{r['status']}]")
    print(f"    SKILL.md: {len(md)} chars → Skill(name={s.name!r})")

print(f"\nLoaded {len(skills_from_search)} skill(s) from semantic search")

### 5d. Path C — MCP endpoint (lazy loading)

This is the most interesting path. Agent Registry exposes itself as an MCP server at:

```
https://bedrock-agentcore.<region>.amazonaws.com/registry/<registry-id>/mcp
```

It exposes a `search_registry_records` tool that any MCP client can call — including a Strands agent at runtime. This means the agent doesn't start with any skills pre-loaded. Instead, it discovers and fetches them **on the fly** based on the task. Self-assembling agents.

For IAM-authenticated access, we use the [`mcp-proxy-for-aws`](https://github.com/aws/mcp-proxy-for-aws) package.

In [ ]:
from strands.tools.mcp import MCPClient
from mcp_proxy_for_aws.client import aws_iam_streamablehttp_client

MCP_URL = f"https://bedrock-agentcore.{REGION}.amazonaws.com/registry/{REGISTRY_ID}/mcp"
print(f"MCP endpoint: {MCP_URL}\n")

# Connect to the registry as an MCP server
mcp_client = MCPClient(lambda: aws_iam_streamablehttp_client(
    endpoint=MCP_URL,
    aws_region=REGION,
    aws_service="bedrock-agentcore",
))

with mcp_client:
    # Discover available MCP tools
    tools = mcp_client.list_tools_sync()
    print(f"MCP tools exposed by registry: {len(tools)}")
    for t in tools:
        print(f"  - {t.tool_name}: {t.tool_spec['description'][:80]}...")

    # Call the search tool via MCP (retry if index not ready)
    mcp_records = []
    for attempt in range(6):
        result = mcp_client.call_tool_sync(
            tool_use_id=f"demo-search-{attempt}",
            name="search_registry_records",
            arguments={
                "searchQuery": "infrastructure security",
                "maxResults": 5,
                "filter": {"descriptorType": {"$eq": "AGENT_SKILLS"}},
            },
        )
        text = result["content"][0]["text"]
        try:
            parsed = json.loads(text) if text else []
            if isinstance(parsed, list):
                mcp_records = parsed
        except json.JSONDecodeError:
            pass
        if mcp_records:
            break
        print(f"  Search index not ready (attempt {attempt + 1}), waiting...")
        time.sleep(15)

    print(f"\nFound {len(mcp_records)} skill(s) via MCP\n")

    for rec in mcp_records:
        md = rec["descriptors"]["agentSkills"]["skillMd"]["inlineContent"]
        skill = Skill.from_content(md)
        print(f"  MCP → Skill.from_content():")
        print(f"    Name:         {skill.name}")
        print(f"    Description:  {skill.description[:60]}...")
        print(f"    Instructions: {len(skill.instructions)} chars")

The MCP path opens up a powerful pattern: an agent that **assembles its own capabilities at runtime**. Instead of hardcoding which skills to load, the agent searches the registry based on what the user asks for, loads the matching skills, and follows their instructions — all in one turn.

---
## Part 6: Summary

Here's what we demonstrated:

| Step | What we did | Key API |
|------|------------|--------|
| Create a skill | Wrote a `SKILL.md` with YAML frontmatter + Markdown instructions | Directory + file |
| Validate it | Parsed with `Skill.from_file()` and checked all fields | `Skill.from_file()` |
| Load from string | Parsed raw Markdown content | `Skill.from_content()` |
| Create in code | Built a Skill object without any files | `Skill()` constructor |
| Bulk load | Scanned a directory for all skills | `Skill.from_directory()` |
| Mix sources | Combined paths + objects in one plugin | `AgentSkills(skills=[...])` |
| Run an agent | Agent activated the right skill and followed its instructions | `Agent(plugins=[plugin])` |
| Add at runtime | Dynamically added a skill to a running plugin | `plugin.set_available_skills()` |
| Inspect activations | Checked which skills the agent used | `plugin.get_activated_skills()` |
| Register in registry | Created a record with `AGENT_SKILLS` descriptor | `create_registry_record()` |
| Direct GET | Fetched by ID, extracted SKILL.md inline | `get_registry_record()` → `Skill.from_content()` |
| Semantic search | Natural language discovery with full descriptors | `search_registry_records()` → `Skill.from_content()` |
| MCP lazy loading | Connected to registry MCP endpoint, searched, loaded skill | `MCPClient` → `Skill.from_content()` |

### The key takeaway

Skills are **instructions, not tools**. They tell the agent *what to do* — the tools let it *do it*. This separation means:

- Skills are portable across any framework that supports the [Agent Skills spec](https://agentskills.io)
- An agent can have hundreds of skills without consuming its context window
- Skills can be shared, versioned, and discovered through registries like [AWS Agent Registry](https://aws.amazon.com/bedrock/agentcore/)
- The full SKILL.md comes back **inline** from every registry API — `GetRegistryRecord`, `SearchRegistryRecords`, and the MCP endpoint all return the complete content, ready for `Skill.from_content()`

---
## Cleanup

In [ ]:
import shutil

# Clean up local files
for d in ["./demo-skills", "./demo-terraform"]:
    if Path(d).exists():
        shutil.rmtree(d)
        print(f"Removed {d}")

# Clean up the registry
try:
    ctrl.delete_registry_record(registryId=REGISTRY_ID, recordId=RECORD_ID)
    print(f"Deleted registry record: {RECORD_ID}")
except Exception as e:
    print(f"Record cleanup: {e}")

try:
    ctrl.delete_registry(registryId=REGISTRY_ID)
    print(f"Deleted registry: {REGISTRY_ID}")
except Exception as e:
    print(f"Registry cleanup: {e}")

print("Done.")